# 📓 Notebook 3: Model Eğitimi

Bu notebook'ta:
- Random Forest, XGBoost ve Logistic Regression eğiteceğiz
- Cross-validation yapacağız
- Model performanslarını karşılaştıracağız
- Baseline sonuçları kaydedeceğiz


In [1]:
import sys
sys.path.append('..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

from src.utils import (compute_metrics, metrics_table, plot_confusion_matrix,
                        plot_roc_curve, plot_feature_importance,
                        save_model, load_model, print_section, FIGURES_DIR)

print('✅ Import başarılı')

✅ Import başarılı


In [2]:
# Notebook 2'den gelen verileri yükle
X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

with open('../data/processed/feature_names.json') as f:
    feature_names = json.load(f)

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Features: {len(feature_names)}')

X_train: (24987, 64)
X_test:  (25013, 64)
Features: 64


## 🤖 Model Tanımları

In [3]:
# Model konfigürasyonları
MODELS = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1,
        n_jobs=-1,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight='balanced',
            n_jobs=-1,
            random_state=42
        ))
    ])
}

print('Modeller tanımlandı:')
for name in MODELS:
    print(f'  - {name}')

Modeller tanımlandı:
  - Random Forest
  - XGBoost
  - Logistic Regression


## 🔄 Cross-Validation

In [4]:
print_section('Cross-Validation (5-Fold Stratified)')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in MODELS.items():
    print(f'  {name} çalışıyor...', end=' ')
    
    scores = cross_val_score(model, X_train, y_train, cv=cv, 
                              scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'F1: {scores.mean():.4f} ± {scores.std():.4f}')

# CV karşılaştırma grafiği
fig, ax = plt.subplots(figsize=(10, 6))
positions = range(len(cv_results))
bp = ax.boxplot([cv_results[n] for n in cv_results], 
                labels=list(cv_results.keys()),
                patch_artist=True)
colors = ['#2196F3', '#FF9800', '#4CAF50']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('Cross-Validation F1 Skorları (5-Fold)', fontsize=13)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cv_results.png', dpi=150, bbox_inches='tight')
plt.show()


════════════════════════════════════════════════════════════
  Cross-Validation (5-Fold Stratified)
════════════════════════════════════════════════════════════

  Random Forest çalışıyor... F1: 0.9229 ± 0.0052
  XGBoost çalışıyor... F1: 0.9389 ± 0.0039
  Logistic Regression çalışıyor... F1: 0.8867 ± 0.0046


## 🎯 Model Eğitimi ve Test

In [5]:
print_section('Model Eğitimi')

trained_models = {}
all_metrics    = []
y_prob_dict    = {}

for name, model in MODELS.items():
    print(f'\n🤖 {name} eğitiliyor...')
    
    # Eğit
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Test
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_prob_dict[name] = y_prob
    
    # Metrik
    metrics = compute_metrics(y_test, y_pred, y_prob, model_name=name)
    all_metrics.append(metrics)
    
    # Confusion matrix
    plot_confusion_matrix(y_test, y_pred, model_name=name)
    
    # Detaylı rapor
    print(f'\nClassification Report - {name}:')
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing']))

# Sonuç tablosu
print_section('Baseline Sonuçlar')
results_df = metrics_table(all_metrics)
print(results_df.to_string())


════════════════════════════════════════════════════════════
  Model Eğitimi
════════════════════════════════════════════════════════════


🤖 Random Forest eğitiliyor...


[2026-02-28 12:37:22] INFO [utils] [Random Forest] ACC=0.9271 F1=0.9267
[2026-02-28 12:37:22] INFO [utils] Confusion matrix kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\figures\cm_random_forest.png



Classification Report - Random Forest:
              precision    recall  f1-score   support

  Legitimate       0.92      0.94      0.93     12448
    Phishing       0.94      0.92      0.93     12565

    accuracy                           0.93     25013
   macro avg       0.93      0.93      0.93     25013
weighted avg       0.93      0.93      0.93     25013


🤖 XGBoost eğitiliyor...


[2026-02-28 12:37:23] INFO [utils] [XGBoost] ACC=0.9392 F1=0.9392
[2026-02-28 12:37:23] INFO [utils] Confusion matrix kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\figures\cm_xgboost.png



Classification Report - XGBoost:
              precision    recall  f1-score   support

  Legitimate       0.93      0.94      0.94     12448
    Phishing       0.94      0.93      0.94     12565

    accuracy                           0.94     25013
   macro avg       0.94      0.94      0.94     25013
weighted avg       0.94      0.94      0.94     25013


🤖 Logistic Regression eğitiliyor...


[2026-02-28 12:37:24] INFO [utils] [Logistic Regression] ACC=0.89 F1=0.8862
[2026-02-28 12:37:24] INFO [utils] Confusion matrix kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\figures\cm_logistic_regression.png



Classification Report - Logistic Regression:
              precision    recall  f1-score   support

  Legitimate       0.86      0.93      0.89     12448
    Phishing       0.92      0.85      0.89     12565

    accuracy                           0.89     25013
   macro avg       0.89      0.89      0.89     25013
weighted avg       0.89      0.89      0.89     25013


════════════════════════════════════════════════════════════
  Baseline Sonuçlar
════════════════════════════════════════════════════════════

                     accuracy  precision  recall      f1     auc
model                                                           
Random Forest          0.9271     0.9370  0.9165  0.9267  0.9807
XGBoost                0.9392     0.9440  0.9343  0.9392  0.9862
Logistic Regression    0.8900     0.9226  0.8526  0.8862  0.9571


In [6]:
# ROC Curves
plot_roc_curve(y_test, y_prob_dict)
print('✅ ROC curves kaydedildi')

[2026-02-28 12:37:24] INFO [utils] ROC curve kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\figures\roc_curves.png


✅ ROC curves kaydedildi


In [7]:
# Feature Importance (RF ve XGBoost için)
for name in ['Random Forest', 'XGBoost']:
    model = trained_models[name]
    
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'named_steps'):  # Pipeline
        estimator = list(model.named_steps.values())[-1]
        if hasattr(estimator, 'feature_importances_'):
            importances = estimator.feature_importances_
        else:
            importances = np.abs(estimator.coef_[0])
    else:
        continue
    
    plot_feature_importance(feature_names, importances, model_name=name, top_n=20)
    print(f'✅ {name} feature importance kaydedildi')

[2026-02-28 12:37:25] INFO [utils] Feature importance kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\figures\fi_random_forest.png


✅ Random Forest feature importance kaydedildi


[2026-02-28 12:37:25] INFO [utils] Feature importance kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\figures\fi_xgboost.png


✅ XGBoost feature importance kaydedildi


In [8]:
# Metrik karşılaştırma bar chart
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, metric in enumerate(metrics_to_plot):
    values = [m[metric] for m in all_metrics]
    names  = [m['model'] for m in all_metrics]
    axes[i].bar(range(len(names)), values, color=colors, alpha=0.8)
    axes[i].set_xticks(range(len(names)))
    axes[i].set_xticklabels(names, rotation=15, ha='right', fontsize=8)
    axes[i].set_title(metric.upper())
    axes[i].set_ylim(0, 1.1)
    for j, v in enumerate(values):
        axes[i].text(j, v + 0.02, f'{v:.3f}', ha='center', fontsize=8)

plt.suptitle('Baseline Model Karşılaştırması', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [9]:
import pickle
from pathlib import Path

save_path = Path(r"C:\Users\emira\OneDrive\Desktop\main projem\data\models")
save_path.mkdir(parents=True, exist_ok=True)

for name, model in trained_models.items():
    safe_name = name.lower().replace(' ', '_')
    path = save_path / f"baseline_{safe_name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(model, f)
    print(f"✅ Kaydedildi: {path}")
    print(trained_models)


✅ Kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\data\models\baseline_random_forest.pkl
{'Random Forest': RandomForestClassifier(class_weight='balanced', max_depth=15,
                       min_samples_leaf=2, min_samples_split=5,
                       n_estimators=200, n_jobs=-1, random_state=42), 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
             